# PySpark Data Transformation Pipeline
Update the configuration in Cell 2, then run the notebook from top to bottom. The pipeline reads a CSV, applies reusable cleansing and type-standardization rules, validates output quality, and writes Parquet.
from __future__ import annotations


In [ ]:
import logging
import os
from pathlib import Path

logging.basicConfig(level=logging.INFO)
from pyspark.sql import functions as F

from common.spark import build_spark_session

MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "data-bucket")
# Define directory paths
BRONZE_DIR = f"s3a://{MINIO_BUCKET}/bronze"
FIXTURES_DIR = Path.cwd() / "tests" / "fixtures"
SILVER_DIR = f"s3a://{MINIO_BUCKET}/silver"
GOLD_DIR = f"s3a://{MINIO_BUCKET}/gold"


## 1. Initialize PySpark Session


In [ ]:
spark = build_spark_session()
print(f"Spark {spark.version} is ready with MinIO integration.")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 100)


In [ ]:
from common.utils import (
    build_create_table_sql,
    load_dataframe,
    store_df_to_table,
    validate_required_columns,
)


## 2. Load Source DataFrames


In [ ]:
df = load_dataframe(spark, f"{BRONZE_DIR}/input.csv", input_format="csv")
df.show()
print(f"Rows: {df.count()}")
df.printSchema()


## Cast dataframe columns, include metadata and descriptions
    spark.sql(f"DESCRIBE TABLE EXTENDED {table_name}").show(truncate=False)
Define all column metadata in a dictionary, then generate SQL dynamically.

In [ ]:
# Define column metadata: type, source name, and description
column_config = {
    "customerId": {
        "source": "customer_id",
        "type": "STRING",
        "description": "Unique identifier for the customer."
    },
    "customerName": {
        "source": "customer_name",
        "type": "STRING",
        "description": "Full name of the customer."
    },
    "amount": {
        "source": "amount",
        "type": "DECIMAL(18,2)",
        "description": "Order amount in the source currency."
    },
    "orderDate": {
        "source": "order_date",
        "type": "DATE",
        "description": "Date the order was placed."
    },
    "region": {
        "source": "region",
        "type": "STRING",
        "description": "Sales region used to partition the data"  # Include partition column description!
    },
}

table_name="my_database.silver_customer_data"
table_comment= "Customer sales data"

create_table_sql = build_create_table_sql(
    table_name=table_name,
    column_config=column_config,
    location=f"{SILVER_DIR}/customer_data",
    table_format="PARQUET",
    partition_columns=["region"],
    table_comment=table_comment,
)

store_df_to_table(spark,
    df=df,
    table_name=table_name,
    column_config=column_config,
    create_table_sql=create_table_sql,
    write_mode="overwrite"
)

In [ ]:
# Verify the inserted data.
spark.sql(f"""
    SELECT *
    FROM {table_name}
""")
# spark.sql(create_t

In [ ]:
spark.sql(
    f"DESCRIBE EXTENDED {table_name}"
).show(truncate=False)


In [ ]:
display(table_df)

In [ ]:
spark.sql(create_table_sql)

In [ ]:
REQUIRED_COLUMNS = ["customer_id", "customer_name", "region", "amount", "order_date"]
validate_required_columns(df, REQUIRED_COLUMNS)
source_df = df


## 3. Inspect and Enforce Schemas


In [ ]:
# Define expected casts as needed for your source, for example:
EXPECTED_TYPES = {"customer_id": "string", "amount": "decimal(18,2)", "event_date": "date"}

typed_df = source_df
for column_name, target_type in EXPECTED_TYPES.items():
    if column_name in typed_df.columns:
        typed_df = typed_df.withColumn(column_name, F.col(column_name).cast(target_type))

typed_df.printSchema()


## 4. Handle Nulls, Duplicates, and Type Casting


In [ ]:
rows_before = typed_df.count()
string_columns = [field.name for field in typed_df.schema.fields if isinstance(field.dataType, StringType)]

clean_df = typed_df
for column_name in string_columns:
    clean_df = clean_df.withColumn(
        column_name,
        F.when(F.trim(F.col(column_name)) == "", F.lit(None))
        .otherwise(F.trim(F.col(column_name)))
    )

non_null_expression = [F.col(column_name).isNotNull() for column_name in clean_df.columns]
if non_null_expression:
    clean_df = clean_df.filter(F.coalesce(*non_null_expression))

deduplication_columns = SOURCE_KEY_COLUMNS or clean_df.columns
clean_df = clean_df.dropDuplicates(deduplication_columns)


## 5. Standardize and Derive Columns


In [ ]:
transformed_df = clean_df

# Optional domain rules; uncomment after setting names that exist in your source.
# transformed_df = transformed_df.withColumn(
#     "event_timestamp", F.to_timestamp("event_timestamp", "yyyy-MM-dd HH:mm:ss")
# )
# transformed_df = transformed_df.withColumn(
#     "amount", F.regexp_replace("amount", r"[^0-9.-]", "").cast("decimal(18,2)")
# )
# transformed_df = transformed_df.withColumn(
#     "status_normalized", F.lower(F.trim(F.regexp_replace("status", r"\s+", " ")))
# )
# transformed_df = transformed_df.withColumn(
#     "is_high_value", F.when(F.col("amount") >= 1000, True).otherwise(False)
# )


## 6. Filter, Join, and Reshape Data


In [ ]:
# Apply source-specific filters, joins, or reshape operations here.
# filtered_df = transformed_df.filter(F.col("status_normalized") == "active")
# enriched_df = filtered_df.join(lookup_df, on="customer_id", how="left")
# pivoted_df = enriched_df.groupBy("region").pivot("category").sum("amount")
# final_df = pivoted_df.select(F.col("region").alias("sales_region"), "*")
final_df = transformed_df


## 7. Aggregate Metrics with GroupBy


In [ ]:
# Example KPI pattern after choosing real grouping and metric columns:
# metrics_df = (
#     final_df.groupBy("region")
#     .agg(
#         F.count("*").alias("row_count"),
#         F.sum("amount").alias("total_amount"),
#         F.avg("amount").alias("average_amount"),
#         F.countDistinct("customer_id").alias("distinct_customers"),
#     )
#     .orderBy(F.desc("total_amount"))
# )
# metrics_df.show(truncate=False)


## 8. Apply Window-Based Transformations


In [ ]:
# Latest-record and running-total examples after setting real columns:
# latest_window = Window.partitionBy("customer_id").orderBy(F.col("event_timestamp").desc())
# latest_df = final_df.withColumn("record_rank", F.row_number().over(latest_window)).filter("record_rank = 1")
# running_window = Window.partitionBy("customer_id").orderBy("event_timestamp").rowsBetween(Window.unboundedPreceding, Window.currentRow)
# running_df = final_df.withColumn("running_amount", F.sum("amount").over(running_window))


## 9. Run Data Quality Checks


In [ ]:
rows_after = final_df.count()
null_count_expressions = [
    F.sum(F.col(column_name).isNull().cast("long")).alias(column_name)
    for column_name in final_df.columns
]
null_counts = final_df.agg(*null_count_expressions) if null_count_expressions else None

quality_report = {
    "rows_before": rows_before,
    "rows_after": rows_after,
    "duplicates_or_empty_rows_removed": rows_before - rows_after,
}
print(quality_report)
if null_counts is not None:
    null_counts.show(truncate=False)

if SOURCE_KEY_COLUMNS:
    duplicate_keys = (
        final_df.groupBy(*SOURCE_KEY_COLUMNS).count().filter(F.col("count") > 1).count()
    )
    assert duplicate_keys == 0, f"Found {duplicate_keys} duplicate business keys"
assert rows_after > 0, "No rows remain after transformation"


## 10. Write Transformed Data to Storage


In [ ]:
(
    final_df.write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)
print(f"Wrote {rows_after:,} rows to {Path(OUTPUT_PATH).resolve()}")

# To partition output, replace the writer above with:
# final_df.write.mode("overwrite").partitionBy("event_date").parquet(OUTPUT_PATH)


## Stop Spark

In [ ]:
spark.stop()
print("Spark session stopped")